# Simulation 7: reproduce in Colab (ChroniCare-AI, protocol v0.6)

**What this notebook does:** downloads the Simulation 7 code from the public repository, **re-runs it from scratch** with the same seeds, and compares the results with the published `sim7_*.csv` files.

- Everything is **simulated**: no patient data are used.
- It takes about **30 minutes** in the environment where it was first run and about **2 h 45 min on free Colab** (2 cores). Do not close the tab or let the computer sleep.
- The last cell states whether the result **reproduces or not**. If there are differences, do not fix them: copy the output as it is and it will be reported in the protocol.

Order: run the cells from top to bottom (Environment, Run, Analyze, Compare).

In [ ]:
# 1. Environment: download the code and print library versions
import os, sys, platform, subprocess
if not os.path.exists('/content/repo'):
    !git clone -q --depth 1 https://github.com/alygallolopez-ship-it/Seminario-doctoral-r6s10714-Metodos-de-Investigacion-cientifica /content/repo
%cd /content/repo/simulation_full_pipeline
import numpy, scipy, pandas, statsmodels, joblib
print('python', platform.python_version(), '| cores:', os.cpu_count())
for m in (numpy, scipy, pandas, statsmodels, joblib):
    print(m.__name__, m.__version__)
print()
print(open('requirements_sim6.txt').read())
print('Commit of the code that will be run:')
!git -C /content/repo log -1 --format='%h %ad %s' --date=short

In [ ]:
# 2. Run the simulation (the long part). Writes R.pkl and RC.pkl to /content/sim7_out
!mkdir -p /content/sim7_out
%time !python run_sim7.py /content/sim7_out

In [ ]:
# 3. Analyze: produce the CSV files from the replicates (same calculations as the published ones)
!python analizar7.py /content/sim7_out /content/sim7_out

In [ ]:
# 4. Compare with the files published in the repository
import glob, hashlib, numpy as np, pandas as pd
rows = []
for f in sorted(glob.glob('/content/repo/simulation_full_pipeline/sim7_*.csv')):
    name = os.path.basename(f); g = '/content/sim7_out/' + name
    if not os.path.exists(g):
        rows.append((name, 'NOT GENERATED', None)); continue
    a, b = pd.read_csv(f), pd.read_csv(g)
    if a.shape != b.shape or list(a.columns) != list(b.columns):
        rows.append((name, 'DIFFERENT SHAPE', None)); continue
    num = a.select_dtypes('number').columns
    dif = float(np.nanmax(np.abs(a[num].values - b[num].values))) if len(num) else 0.0
    txt = all((a[c].astype(str) == b[c].astype(str)).all() for c in a.columns if c not in num)
    rows.append((name, 'IDENTICAL' if (dif < 1e-9 and txt) else ('NEARLY IDENTICAL (diff <= 1e-6)' if (dif < 1e-6 and txt) else 'DIFFERENT'), dif))
res = pd.DataFrame(rows, columns=['file', 'result', 'max_numeric_difference'])
print(res.to_string(index=False))
print()
print('SHA-256 fingerprints of the files you just generated:')
for g in sorted(glob.glob('/content/sim7_out/sim7_*.csv')):
    print(hashlib.sha256(open(g, 'rb').read()).hexdigest()[:16], os.path.basename(g))
ok = all(r[1] in ('IDENTICAL', 'NEARLY IDENTICAL (diff <= 1e-6)') for r in rows) and len(rows) > 0
print()
print('REPRODUCED' if ok else 'NOT REPRODUCED: copy this full output and send it without changes')

In [ ]:
# 5. Key figures cited in the protocol (for a visual check)
vc = pd.read_csv('/content/sim7_out/sim7_valores_criticos.csv')
print(vc[['n', 'resultado', 'vc_a025']].round(3).to_string(index=False))
tam = pd.read_csv('/content/sim7_out/sim7_tamano.csv'); print(); print(tam.head(8).round(3).to_string(index=False))

**When it finishes:** take a screenshot of the output of cell 4 (or copy the text) and send it to Claude. The protocol's AI-use declaration will then state that Simulation 7 was reproduced by the researcher in Google Colab.